# Notebook 11: 在 Qwen3.5 上部署自定义 Triton GEMM

## 本章内容

前面 10 章我们学了：
- 如何写 Triton kernel（01）
- 如何注册为 CustomOp（02）
- 如何替换模型算子（04）
- vLLM 中有哪些矩阵乘法（06-09）
- TP 通信与 GEMM 如何配合（10）

**现在到了最终实战：在 Qwen3.5 模型上，用自己写的 Triton 矩阵乘法 kernel 替换 vLLM 默认的 cuBLAS GEMM，让它真正跑起来。**

这一章会覆盖三条替换路径，从最简单到最底层：

| 路径 | 替换什么 | 机制 | 难度 |
|------|---------|------|------|
| A | 激活函数（SiluAndMul） | CustomOp.register_oot | ★☆☆ |
| B | 整个线性层（ColumnParallelLinear） | PluggableLayer.register_oot | ★★☆ |
| C | GEMM 内核（UnquantizedLinearMethod.apply） | 猴子补丁 / 自定义 QuantizeMethod | ★★★ |

> **前置知识：** Notebook 01-05（Triton 基础 + CustomOp + 替换机制），Notebook 06（矩阵乘法全景）

## 1. Qwen3.5 模型架构速览

先搞清楚我们的"手术对象"长什么样。Qwen3.5 是一个混合架构模型，每层可能是：
- **full_attention 层** — 标准 Transformer 自注意力（QKV + O 投影）
- **linear_attention 层** — GatedDeltaNet（线性注意力变体）

两种层都包含 MLP，而 MLP 就是我们要替换 GEMM 的主要目标。

### Qwen3.5 Dense 模型的 MLP 结构

```
Qwen3_5DecoderLayer
├── self_attn / linear_attn
│   ├── qkv_proj  → QKVParallelLinear      (ColumnParallel)
│   └── o_proj    → RowParallelLinear
├── mlp (Qwen2MLP)
│   ├── gate_up_proj → MergedColumnParallelLinear  ← 目标 1
│   │   内部: Y = X @ [W_gate, W_up]   (一次 GEMM 输出两个矩阵)
│   ├── act_fn → SiluAndMul(CustomOp)               ← 目标 2
│   └── down_proj → RowParallelLinear               ← 目标 3
├── input_layernorm
└── post_attention_layernorm
```

### 关键调用链

```
MLP.forward(x):
    gate_up, _ = self.gate_up_proj(x)     # GEMM: cuBLAS (torch.nn.functional.linear)
    x = self.act_fn(gate_up)               # Triton/CUDA kernel (SiluAndMul)
    x, _ = self.down_proj(x)               # GEMM: cuBLAS
    return x
```

说白了，MLP 里面有 **两次 GEMM**（gate_up_proj 和 down_proj）和 **一次激活函数**。

> Source: `vllm/model_executor/models/qwen2.py:107-111`（Qwen2MLP.forward）
> Source: `vllm/model_executor/models/qwen3_5.py:248-259`（Qwen3_5DecoderLayer.__init__）

## 2. vLLM 中 GEMM 的完整调用链

要替换 GEMM，得先知道 GEMM 是怎么被调用的。让我们追踪一次 `gate_up_proj(x)` 的完整路径：

```
gate_up_proj(x)                              # MergedColumnParallelLinear.forward()
  → self.quant_method.apply(self, x, bias)   # UnquantizedLinearMethod.apply()
    → dispatch_unquantized_gemm()(layer, x, layer.weight, bias)
      → default_unquantized_gemm(layer, x, weight, bias)
        → torch.nn.functional.linear(x, weight, bias)   # 最终调 cuBLAS
```

四层调用栈！从模型层一直到 cuBLAS。每一层都是一个替换点：

| 层级 | 类/函数 | 替换方式 | 影响范围 |
|------|---------|---------|---------|
| L1 | `MergedColumnParallelLinear` | PluggableLayer.register_oot | 替换整个层 |
| L2 | `UnquantizedLinearMethod.apply` | 自定义 QuantizeMethod | 替换 GEMM 调度 |
| L3 | `dispatch_unquantized_gemm` | 猴子补丁 | 替换底层 GEMM |
| L4 | `torch.nn.functional.linear` | PyTorch 不建议替换 | 太底层了 |

> Source: `vllm/model_executor/layers/linear.py:604`（ColumnParallelLinear.forward）
> Source: `vllm/model_executor/layers/linear.py:238-250`（UnquantizedLinearMethod.apply）
> Source: `vllm/model_executor/layers/utils.py:298-304`（dispatch_unquantized_gemm）

## 3. 编写我们的 Triton GEMM Kernel

这是本教程的核心——一个生产级的 Triton 矩阵乘法。我们会从 Notebook 06-09 学到的知识出发，写一个支持：
- FP16 计算
- 分块 tiling（BLOCK_M, BLOCK_N, BLOCK_K）
- 自动调优（@triton.autotune）

的 GEMM kernel。

In [ ]:
import torch
import triton
import triton.language as tl

# ============================================================
# 我们的 Triton GEMM Kernel
# ============================================================

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 32, 'GROUP_SIZE_M': 8}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 64,  'BLOCK_K': 32, 'GROUP_SIZE_M': 8}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 128, 'BLOCK_K': 32, 'GROUP_SIZE_M': 8}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 64,  'BLOCK_K': 64, 'GROUP_SIZE_M': 8}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 64,  'BLOCK_K': 32, 'GROUP_SIZE_M': 8}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 32,  'BLOCK_N': 32,  'BLOCK_K': 64, 'GROUP_SIZE_M': 4}, num_warps=4, num_stages=3),
    ],
    key=['M', 'N', 'K'],
)
@triton.jit
def our_matmul_kernel(
    # 指针
    A_ptr, B_ptr, C_ptr,
    # 矩阵维度
    M, N, K,
    # strides
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    # 分块参数
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):
    """
    计算 C = A @ B
    A: (M, K), B: (K, N), C: (M, N)

    核心思路和 Notebook 06-09 的 GEMM 一样：
    1. 每个 program 负责计算 C 的一个 (BLOCK_M, BLOCK_N) 块
    2. 沿 K 维度循环累加
    3. GROUP_SIZE_M 做 swizzle 提高 L2 cache 命中率
    """
    # --- Program ID 与 swizzle ---
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    # L2 cache 友好的 swizzle（和 vLLM fused_moe_kernel 的 GROUP_SIZE_M 一样）
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # --- 计算指针偏移 ---
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    # A 块的指针: (BLOCK_M, BLOCK_K)
    a_ptrs = A_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    # B 块的指针: (BLOCK_K, BLOCK_N)
    b_ptrs = B_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    # --- 主循环: 沿 K 维度累加 ---
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k_start in range(0, K, BLOCK_K):
        # 加载 A 块
        a = tl.load(a_ptrs,
                     mask=(offs_m[:, None] < M) & ((k_start + offs_k[None, :]) < K),
                     other=0.0)
        # 加载 B 块
        b = tl.load(b_ptrs,
                     mask=((k_start + offs_k[:, None]) < K) & (offs_n[None, :] < N),
                     other=0.0)
        # 核心: tl.dot — 这就是 Tensor Core 上的矩阵乘法
        acc = tl.dot(a, b, acc)
        # 移动 K 指针
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # --- 存储结果 ---
    c = acc.to(tl.float16)
    c_ptrs = C_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, c, mask=mask)


def our_triton_linear(x: torch.Tensor, weight: torch.Tensor,
                      bias: torch.Tensor | None = None) -> torch.Tensor:
    """
    替换 torch.nn.functional.linear 的 Triton 版本。

    注意 F.linear 的约定: Y = X @ W^T + bias
    所以 A=X (M,K), B=W^T (K,N)
    """
    assert x.is_cuda and weight.is_cuda
    # x: (*, K), weight: (N, K) → Y: (*, N)
    x_2d = x.view(-1, x.shape[-1])   # (M, K)
    M, K = x_2d.shape
    N = weight.shape[0]

    # 输出
    output = torch.empty((M, N), device=x.device, dtype=x.dtype)

    # weight 是 (N, K)，我们需要 (K, N)，所以转置
    weight_t = weight.t().contiguous()

    # Grid: 每个 program 计算一个 (BLOCK_M, BLOCK_N) 的输出块
    grid = lambda meta: (triton.cdiv(M, meta['BLOCK_M']) * triton.cdiv(N, meta['BLOCK_N']),)

    our_matmul_kernel[grid](
        x_2d, weight_t, output,
        M, N, K,
        x_2d.stride(0), x_2d.stride(1),
        weight_t.stride(0), weight_t.stride(1),
        output.stride(0), output.stride(1),
    )

    if bias is not None:
        output += bias

    return output.view(*x.shape[:-1], N)


print("\u2713 our_triton_linear 定义完成")

### 3.1 验证正确性

在替换任何东西之前，先确保我们的 kernel 和 `torch.nn.functional.linear` 算出一样的结果。

In [ ]:
# --- 正确性验证 ---
torch.manual_seed(42)
M, K, N = 1024, 4096, 11008  # 典型 Qwen3 MLP 尺寸
x = torch.randn(M, K, device='cuda', dtype=torch.float16)
weight = torch.randn(N, K, device='cuda', dtype=torch.float16)
bias = torch.randn(N, device='cuda', dtype=torch.float16)

# 参考实现
y_ref = torch.nn.functional.linear(x, weight, bias)

# 我们的实现
y_ours = our_triton_linear(x, weight, bias)

# 比较
max_diff = (y_ref - y_ours).abs().max().item()
mean_diff = (y_ref - y_ours).abs().mean().item()
rel_err = mean_diff / y_ref.abs().mean().item()

print(f"最大绝对误差: {max_diff:.6f}")
print(f"平均绝对误差: {mean_diff:.6f}")
print(f"相对误差:     {rel_err:.2e}")
assert torch.allclose(y_ref, y_ours, atol=1e-1, rtol=1e-2), \
    f"误差过大! max_diff={max_diff}"
print("\u2713 Triton GEMM 正确性验证通过！")

## 4. 路径 A: 通过 CustomOp 替换激活函数（最简单）

这是最温和的入门——不动 GEMM，只替换 SiluAndMul 激活函数。但它展示了完整的 OOT 注册机制。

### 原理

vLLM 的 SiluAndMul 是一个 `CustomOp`：

```python
@CustomOp.register("silu_and_mul")
class SiluAndMul(CustomOp):
    def forward_native(self, x):
        d = x.shape[-1] // 2
        return F.silu(x[..., :d]) * x[..., d:]

    def forward_cuda(self, x):
        # 调用 CUDA kernel: vllm._C.ops.silu_and_mul(out, x)
        ...
```

我们可以用 `@CustomOp.register_oot("SiluAndMul")` 来替换它——注意这里用的是 **类名**，不是 register 的 op_name。

> Source: `vllm/model_executor/layers/activation.py:117-118`
> Source: `vllm/model_executor/custom_op.py:289-310`（register_oot 机制）

In [ ]:
from vllm.model_executor.custom_op import CustomOp
from vllm.model_executor.layers.activation import SiluAndMul
import torch.nn.functional as F

# ============================================================
# 用 Triton 实现 SiluAndMul
# ============================================================

@triton.jit
def silu_and_mul_kernel(
    X_ptr, Out_ptr,
    N,  # 输出维度 (= 输入维度 / 2)
    stride_x, stride_out,
    BLOCK_SIZE: tl.constexpr,
):
    """Fused SiLU(x[:d]) * x[d:] kernel"""
    row_id = tl.program_id(0)
    offs = tl.arange(0, BLOCK_SIZE)
    mask = offs < N

    # x 的前半部分 (gate) 和后半部分 (up)
    gate = tl.load(X_ptr + row_id * stride_x + offs, mask=mask)
    up = tl.load(X_ptr + row_id * stride_x + N + offs, mask=mask)

    # SiLU(gate) = gate * sigmoid(gate)
    silu_gate = gate * tl.sigmoid(gate)

    # 输出 = SiLU(gate) * up
    result = silu_gate * up
    tl.store(Out_ptr + row_id * stride_out + offs, result, mask=mask)


class OurSiluAndMul(SiluAndMul):
    """用 Triton 替换 vLLM 原生的 SiluAndMul"""

    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        d = x.shape[-1] // 2
        output = torch.empty(*x.shape[:-1], d, device=x.device, dtype=x.dtype)

        n_rows = x.numel() // x.shape[-1]
        BLOCK_SIZE = triton.next_power_of_2(d)

        silu_and_mul_kernel[(n_rows,)](
            x, output,
            d,
            x.stride(-2) if x.dim() > 1 else x.shape[-1],
            output.stride(-2) if output.dim() > 1 else d,
            BLOCK_SIZE=BLOCK_SIZE,
        )
        return output


# --- 验证 ---
x_test = torch.randn(32, 22016, device='cuda', dtype=torch.float16)  # 11008 * 2
y_ref = F.silu(x_test[..., :11008]) * x_test[..., 11008:]
y_ours = OurSiluAndMul().forward_cuda(x_test)
assert torch.allclose(y_ref, y_ours, atol=1e-2), "SiluAndMul 验证失败"
print("\u2713 OurSiluAndMul 正确性验证通过！")

### 4.1 注册到 vLLM

有了实现，注册只需一行装饰器：

```python
# 方式 1: 装饰器
@CustomOp.register_oot(name="SiluAndMul")
class OurSiluAndMul(SiluAndMul):
    ...

# 方式 2: 手动注册（适合批量注册）
CustomOp.register_oot(_decorated_op_cls=OurSiluAndMul, name="SiluAndMul")
```

注册后，任何代码 `SiluAndMul()` 都会自动实例化 `OurSiluAndMul`，因为 `CustomOp.__new__()` 会检查 `op_registry_oot`。

**注意：** 注册时的 `name` 必须是原始类的 **类名**（`"SiluAndMul"`），不是 register 的 op_name（`"silu_and_mul"`）。这是因为 `__new__` 用 `cls.__name__` 查找。

> Source: `vllm/model_executor/custom_op.py:99-118`（CustomOp.__new__）

## 5. 路径 B: 通过 PluggableLayer 替换线性层

这是**替换 GEMM 的推荐方式**——替换整个 `ColumnParallelLinear` 或 `RowParallelLinear`。

### PluggableLayer vs CustomOp

| 特性 | CustomOp | PluggableLayer |
|------|----------|----------------|
| 用途 | 算子级别（SiLU, RMSNorm） | 层级别（Linear, MLA） |
| 平台分发 | forward_cuda/forward_hip/... | 无（自己写 forward） |
| 状态 | 无参数 | 有参数（weights） |
| 替换机制 | `register_oot` → `__new__` 拦截 | `register_oot` → `__new__` 拦截 |

`ColumnParallelLinear` 继承自 `PluggableLayer`（通过 `LinearBase`），所以我们可以用 `PluggableLayer.register_oot` 来替换它。

> Source: `vllm/model_executor/layers/linear.py:253`（`class LinearBase(PluggableLayer)`）
> Source: `vllm/model_executor/layers/linear.py:428`（`@PluggableLayer.register("column_parallel_linear")`）

In [ ]:
from vllm.model_executor.custom_op import PluggableLayer
from vllm.model_executor.layers.linear import (
    MergedColumnParallelLinear,
    RowParallelLinear,
)

# ============================================================
# 替换 MergedColumnParallelLinear，使用我们的 Triton GEMM
# ============================================================

class OurMergedColumnParallelLinear(MergedColumnParallelLinear):
    """
    继承原始的 MergedColumnParallelLinear，
    只覆盖 forward 中的 GEMM 部分。

    原始 forward 调用 self.quant_method.apply(self, input_, bias)，
    最终走到 torch.nn.functional.linear。
    我们直接覆盖 forward，用 our_triton_linear 替换。
    """

    def forward(self, input_):
        bias = self.bias if not self.skip_bias_add else None

        # --- 替换点：用 Triton GEMM 代替 cuBLAS ---
        output_parallel = our_triton_linear(input_, self.weight, bias)

        if self.gather_output and self.tp_size > 1:
            from vllm.distributed import tensor_model_parallel_all_gather
            output = tensor_model_parallel_all_gather(output_parallel)
        else:
            output = output_parallel

        if not self.return_bias:
            return output
        output_bias = self.bias if self.skip_bias_add else None
        return output, output_bias


class OurRowParallelLinear(RowParallelLinear):
    """同理替换 RowParallelLinear"""

    def forward(self, input_):
        if self.input_is_parallel:
            input_parallel = input_
        else:
            from vllm.distributed import (
                get_tensor_model_parallel_world_size,
                tensor_model_parallel_all_gather,
            )
            tp_size = get_tensor_model_parallel_world_size()
            if tp_size > 1:
                splitted = tensor_model_parallel_all_gather(input_)
                input_parallel = splitted
            else:
                input_parallel = input_

        # --- 替换点：用 Triton GEMM ---
        output_parallel = our_triton_linear(input_parallel, self.weight)

        if self.reduce_results and self.tp_size > 1:
            from vllm.distributed import tensor_model_parallel_all_reduce
            output_ = tensor_model_parallel_all_reduce(output_parallel)
        else:
            output_ = output_parallel

        if not self.skip_bias_add:
            if self.bias is not None:
                output = output_ + self.bias
            else:
                output = output_
            output_bias = None
        else:
            output = output_
            output_bias = self.bias

        if not self.return_bias:
            return output
        return output, output_bias


print("\u2713 OurMergedColumnParallelLinear / OurRowParallelLinear 定义完成")

### 5.1 注册 PluggableLayer

注册方式和 CustomOp 类似：

```python
# 注册替换 — 之后所有 MergedColumnParallelLinear() 都会变成我们的版本
PluggableLayer.register_oot(
    _decorated_layer_cls=OurMergedColumnParallelLinear,
    name="MergedColumnParallelLinear"
)
PluggableLayer.register_oot(
    _decorated_layer_cls=OurRowParallelLinear,
    name="RowParallelLinear"
)
```

**注意事项：**
1. `name` 必须是原始类名（不是 register 的 op_name）
2. 注册必须在模型实例化 **之前** 完成（因为 `__new__` 只在创建对象时检查）
3. 替换的类必须继承原始类，否则权重加载会出问题
4. 这会替换 **所有模型的所有层**，不只是 Qwen3.5

### PluggableLayer.__new__ 的工作流程

```
PluggableLayer.__new__(cls)
  → layer_class_name = cls.__name__   # e.g., "MergedColumnParallelLinear"
  → 检查 op_registry_oot[layer_class_name]
  → 如果找到 → 用 OOT 类创建实例
  → 如果没找到 → 用原始类创建实例
```

> Source: `vllm/model_executor/custom_op.py:37-56`（PluggableLayer.__new__）

## 6. 路径 C: 替换底层 GEMM 分发（最精确）

如果你只想替换 GEMM 内核本身，不想改变层的其他逻辑（TP 通信、权重加载等），可以直接替换 `UnquantizedLinearMethod.apply` 或 `dispatch_unquantized_gemm`。

### 方式 1: 猴子补丁 dispatch_unquantized_gemm

这是最简单的底层替换——一行代码搞定：

In [ ]:
import vllm.model_executor.layers.utils as layer_utils

# 保存原始函数
original_dispatch = layer_utils.dispatch_unquantized_gemm

def our_dispatch_unquantized_gemm():
    """返回我们的 Triton GEMM 函数"""
    def triton_gemm(layer, x, weight, bias=None):
        return our_triton_linear(x, weight, bias)
    return triton_gemm

# --- 替换 ---
# layer_utils.dispatch_unquantized_gemm = our_dispatch_unquantized_gemm

# 取消注释上面那行就能全局替换所有未量化的 GEMM
# 注意：这会影响所有模型的所有线性层！

print("dispatch_unquantized_gemm 猴子补丁已准备好（未激活）")
print(f"原始函数: {original_dispatch}")
print(f"替换函数: {our_dispatch_unquantized_gemm}")

### 方式 2: 自定义 LinearMethod（更正规）

如果你不想用猴子补丁，可以实现一个自定义的 `LinearMethodBase` 子类：

In [ ]:
from vllm.model_executor.layers.linear import (
    LinearMethodBase, UnquantizedLinearMethod
)

class TritonLinearMethod(UnquantizedLinearMethod):
    """用 Triton GEMM 替换 cuBLAS 的 LinearMethod"""

    def apply(
        self,
        layer: torch.nn.Module,
        x: torch.Tensor,
        bias: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # 直接用我们的 Triton GEMM
        return our_triton_linear(x, layer.weight, bias)


# --- 验证 ---
# 模拟一个有 weight 属性的层
class FakeLayer(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = torch.randn(1024, 4096, device='cuda', dtype=torch.float16)

method = TritonLinearMethod()
fake_layer = FakeLayer()
x_test = torch.randn(32, 4096, device='cuda', dtype=torch.float16)

y_triton = method.apply(fake_layer, x_test)
y_ref = torch.nn.functional.linear(x_test, fake_layer.weight)
assert torch.allclose(y_ref, y_triton, atol=1e-1, rtol=1e-2)
print(f"\u2713 TritonLinearMethod 验证通过！输出形状: {y_triton.shape}")

## 7. 完整实战：在 Qwen3.5 上启用自定义 GEMM

现在我们把所有东西组装起来。在实际部署中，你需要：

### 步骤 1: 创建一个插件包

```
my_triton_gemm_plugin/
├── setup.py
├── my_triton_gemm/
│   ├── __init__.py          # 入口，注册所有替换
│   ├── kernels.py           # Triton kernel 定义
│   └── layers.py            # 替换层定义
```

### 步骤 2: __init__.py 中注册

```python
# my_triton_gemm/__init__.py
from vllm.model_executor.custom_op import PluggableLayer

from .layers import OurMergedColumnParallelLinear, OurRowParallelLinear

# 注册替换
PluggableLayer.register_oot(
    _decorated_layer_cls=OurMergedColumnParallelLinear,
    name="MergedColumnParallelLinear"
)
PluggableLayer.register_oot(
    _decorated_layer_cls=OurRowParallelLinear,
    name="RowParallelLinear"
)

print("\u2713 Triton GEMM 插件已注册")
```

### 步骤 3: 启动 vLLM 时加载插件

```bash
# 方式 1: 通过环境变量
VLLM_PLUGINS=my_triton_gemm vllm serve Qwen/Qwen3-8B

# 方式 2: pip install 后自动发现（推荐）
pip install -e ./my_triton_gemm_plugin
vllm serve Qwen/Qwen3-8B
```

### 步骤 4: 验证替换生效

```python
from vllm.model_executor.custom_op import op_registry_oot
print("已注册的 OOT 替换:")
for name, cls in op_registry_oot.items():
    print(f"  {name} → {cls.__name__}")
```

### 7.1 完整的 setup.py（支持 pip install 自动发现）

vLLM 使用 Python 的 entry_points 机制来自动发现插件：

In [ ]:
# --- 这是 setup.py 的内容（不要在 notebook 中运行） ---
setup_py_content = """
from setuptools import setup, find_packages

setup(
    name="my-triton-gemm-plugin",
    version="0.1.0",
    packages=find_packages(),
    install_requires=["vllm", "triton"],
    entry_points={
        "vllm.general_plugins": [
            "my_triton_gemm = my_triton_gemm:register",
        ],
    },
)
"""

init_py_content = """
from vllm.model_executor.custom_op import PluggableLayer
from .layers import OurMergedColumnParallelLinear, OurRowParallelLinear

def register():
    \"\"\"vLLM 插件入口点，启动时自动调用\"\"\"
    PluggableLayer.register_oot(
        _decorated_layer_cls=OurMergedColumnParallelLinear,
        name="MergedColumnParallelLinear"
    )
    PluggableLayer.register_oot(
        _decorated_layer_cls=OurRowParallelLinear,
        name="RowParallelLinear"
    )
    print("[my_triton_gemm] Triton GEMM plugin registered!")
"""

print("=== setup.py ===")
print(setup_py_content)
print("=== __init__.py ===")
print(init_py_content)

## 8. Qwen3.5 特殊考量

Qwen3.5 不是一个普通的 Dense 模型，它有几个特殊之处需要注意：

### 8.1 混合架构：full_attention + linear_attention

```
Qwen3.5 的层类型由 config.layer_types 决定：
  layer_types = ["full_attention", "linear_attention", "full_attention", ...]

full_attention 层:
  self_attn → Qwen3NextAttention
    qkv_proj: QKVParallelLinear    ← 你的 Triton GEMM 会替换这里
    o_proj:   RowParallelLinear     ← 和这里

linear_attention 层:
  linear_attn → Qwen3_5GatedDeltaNet
    in_proj_qkvz: MergedColumnParallelLinear  ← 和这里
    in_proj_ba:   MergedColumnParallelLinear  ← 和这里
    out_proj:     RowParallelLinear            ← 和这里
```

**关键点：** 如果你替换了 `MergedColumnParallelLinear`，两种层类型的投影都会被替换。

### 8.2 MoE 变体（Qwen3.5-MoE）

Qwen3.5 也有 MoE 版本，MLP 变成 `Qwen3NextSparseMoeBlock`。MoE 层的 GEMM 走 `fused_moe_kernel`（Triton 实现），**不经过 ColumnParallelLinear**。所以替换 Linear 层不会影响 MoE 的 Expert GEMM。

### 8.3 替换范围总结

| 组件 | 替换 Linear 是否影响 | 原因 |
|------|---------------------|------|
| Attention QKV | ✅ 是 | QKVParallelLinear 继承 ColumnParallel |
| Attention O | ✅ 是 | RowParallelLinear |
| Dense MLP gate_up | ✅ 是 | MergedColumnParallelLinear |
| Dense MLP down | ✅ 是 | RowParallelLinear |
| MoE Expert GEMM | ❌ 否 | 走 fused_moe_kernel，独立路径 |
| GDN in_proj | ✅ 是 | MergedColumnParallelLinear |
| GDN out_proj | ✅ 是 | RowParallelLinear |
| Embedding | ❌ 否 | VocabParallelEmbedding，不是 Linear |

> Source: `vllm/model_executor/models/qwen3_5.py:207-261`（Qwen3_5DecoderLayer）

## 9. 性能基准测试

替换之后最重要的问题：**快了还是慢了？**

### 9.1 单 GEMM 对比

In [ ]:
import time

def benchmark_gemm(fn, x, weight, bias=None, warmup=10, repeat=100):
    """benchmark 一个 GEMM 函数"""
    # warmup
    for _ in range(warmup):
        fn(x, weight, bias)
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(repeat):
        fn(x, weight, bias)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / repeat
    return elapsed

# 测试不同尺寸
sizes = [
    ("Qwen3-8B MLP gate_up", 1, 4096, 22016),    # batch=1 (decode)
    ("Qwen3-8B MLP gate_up", 128, 4096, 22016),   # batch=128 (prefill)
    ("Qwen3-8B MLP gate_up", 1024, 4096, 22016),  # batch=1024
    ("Qwen3-8B MLP down",    1, 11008, 4096),
    ("Qwen3-8B MLP down",    128, 11008, 4096),
    ("Qwen3-8B Attn QKV",    128, 4096, 6144),    # Q + K + V packed
]

print(f"{'名称':<25} {'M':>6} {'K':>6} {'N':>6} {'cuBLAS(us)':>12} {'Triton(us)':>12} {'比值':>8}")
print("-" * 85)

for name, M, K, N in sizes:
    x = torch.randn(M, K, device='cuda', dtype=torch.float16)
    w = torch.randn(N, K, device='cuda', dtype=torch.float16)

    t_cublas = benchmark_gemm(torch.nn.functional.linear, x, w) * 1e6
    t_triton = benchmark_gemm(our_triton_linear, x, w) * 1e6
    ratio = t_triton / t_cublas

    print(f"{name:<25} {M:>6} {K:>6} {N:>6} {t_cublas:>10.1f}us {t_triton:>10.1f}us {ratio:>7.2f}x")

print()
print("比值 < 1.0 表示 Triton 更快，> 1.0 表示 cuBLAS 更快")
print("提示：batch=1 时 cuBLAS 通常更快（它对小 M 做了特殊优化）")

## 10. 进阶：选择性替换（只替换特定层）

全局替换太暴力了？我们可以做更精细的控制——只替换 MLP 的 GEMM，不碰 Attention 的。

### 方式 1: 模型加载后手动替换

In [ ]:
def selective_replace_mlp_gemm(model):
    """只替换 MLP 层的 GEMM，不碰 Attention"""
    replaced = 0
    for name, module in model.named_modules():
        # 只替换 mlp 下面的线性层
        if '.mlp.' not in name:
            continue

        if isinstance(module, MergedColumnParallelLinear):
            # 修改 forward 方法
            original_forward = module.forward

            def make_triton_forward(orig_module):
                def triton_forward(input_):
                    bias = orig_module.bias if not orig_module.skip_bias_add else None
                    output = our_triton_linear(input_, orig_module.weight, bias)
                    if not orig_module.return_bias:
                        return output
                    output_bias = orig_module.bias if orig_module.skip_bias_add else None
                    return output, output_bias
                return triton_forward

            module.forward = make_triton_forward(module)
            replaced += 1

        elif isinstance(module, RowParallelLinear):
            original_forward = module.forward

            def make_triton_row_forward(orig_module):
                def triton_forward(input_):
                    output_parallel = our_triton_linear(input_, orig_module.weight)
                    if orig_module.reduce_results and orig_module.tp_size > 1:
                        from vllm.distributed import tensor_model_parallel_all_reduce
                        output_ = tensor_model_parallel_all_reduce(output_parallel)
                    else:
                        output_ = output_parallel
                    if not orig_module.skip_bias_add:
                        output = output_ + orig_module.bias if orig_module.bias is not None else output_
                        output_bias = None
                    else:
                        output = output_
                        output_bias = orig_module.bias
                    if not orig_module.return_bias:
                        return output
                    return output, output_bias
                return triton_forward

            module.forward = make_triton_row_forward(module)
            replaced += 1

    return replaced


# 使用方式（在模型加载后调用）：
# n = selective_replace_mlp_gemm(model)
# print(f"已替换 {n} 个 MLP 线性层")

print("selective_replace_mlp_gemm 函数已定义")
print("使用方式: n = selective_replace_mlp_gemm(model)")

### 方式 2: 在 OOT 层中做条件判断

更优雅的方式——在替换层内部检查自己的 prefix，决定是否使用 Triton：

In [ ]:
class SmartColumnParallelLinear(MergedColumnParallelLinear):
    """
    智能线性层：只在 MLP 路径使用 Triton GEMM，
    Attention 路径仍然使用 cuBLAS。
    """

    def forward(self, input_):
        bias = self.bias if not self.skip_bias_add else None

        # 检查这个层是否在 MLP 路径中
        use_triton = hasattr(self, 'prefix') and '.mlp.' in getattr(self, 'prefix', '')

        if use_triton:
            output_parallel = our_triton_linear(input_, self.weight, bias)
        else:
            # 回退到原始 cuBLAS 路径
            assert self.quant_method is not None
            output_parallel = self.quant_method.apply(self, input_, bias)

        if self.gather_output and self.tp_size > 1:
            from vllm.distributed import tensor_model_parallel_all_gather
            output = tensor_model_parallel_all_gather(output_parallel)
        else:
            output = output_parallel

        if not self.return_bias:
            return output
        output_bias = self.bias if self.skip_bias_add else None
        return output, output_bias


print("SmartColumnParallelLinear: 只在 .mlp. 路径使用 Triton")
print("prefix 示例: model.layers.0.mlp.gate_up_proj → 使用 Triton")
print("prefix 示例: model.layers.0.self_attn.qkv_proj → 使用 cuBLAS")

## 11. 端到端部署指南

### 完整步骤清单

```bash
# 1. 创建插件目录
mkdir -p my_triton_gemm_plugin/my_triton_gemm

# 2. 将 kernels.py 和 layers.py 复制进去
# （从本 notebook 的 Cell 3 和 Cell 7 提取代码）

# 3. 写 setup.py（Cell 12）和 __init__.py（Cell 12）

# 4. 安装
cd my_triton_gemm_plugin
pip install -e .

# 5. 启动 vLLM（自动加载插件）
vllm serve Qwen/Qwen3-8B --dtype float16

# 6. 验证替换生效
python -c "
from vllm.model_executor.custom_op import op_registry_oot
print('OOT 替换:', dict(op_registry_oot))
"
```

### 调试技巧

| 问题 | 排查方式 |
|------|---------|
| 插件没加载 | 检查 `entry_points`，运行 `pip show my-triton-gemm-plugin` |
| 精度下降 | 对比 Triton vs cuBLAS 输出，检查 FP16 累加误差 |
| 性能变差 | 用 `nsys profile` 看 kernel launch overhead |
| OOM | Triton JIT 编译消耗额外 GPU 内存，`--gpu-memory-utilization 0.85` |
| 张量形状不匹配 | 检查 weight 是否 contiguous，Triton 要求连续内存 |

### 重要限制

1. **量化模型不走这条路径** — FP8/INT8 模型使用 CUTLASS/Marlin，不经过 `UnquantizedLinearMethod`
2. **MoE Expert GEMM 独立** — `fused_moe_kernel` 是单独的 Triton kernel，需要另外替换
3. **torch.compile 兼容性** — Triton kernel 作为 custom op 时可能影响图编译
4. **多 GPU** — 替换层必须正确处理 TP 通信（继承原始层可以自动获得）

## 源码映射表

| 本 Notebook | vLLM 原始源码 | 说明 |
|-------------|--------------|------|
| `our_matmul_kernel` | `torch.nn.functional.linear` → cuBLAS | 替换目标 |
| `our_triton_linear` | `vllm/model_executor/layers/utils.py:113-119` `default_unquantized_gemm` | F.linear 的包装 |
| `OurSiluAndMul` | `vllm/model_executor/layers/activation.py:117-118` `SiluAndMul` | 路径 A |
| `OurMergedColumnParallelLinear` | `vllm/model_executor/layers/linear.py:428,626` `MergedColumnParallelLinear` | 路径 B |
| `OurRowParallelLinear` | `vllm/model_executor/layers/linear.py:1307` `RowParallelLinear` | 路径 B |
| `TritonLinearMethod` | `vllm/model_executor/layers/linear.py:200-250` `UnquantizedLinearMethod` | 路径 C |
| `selective_replace_mlp_gemm` | — | 选择性替换工具 |
| `SmartColumnParallelLinear` | — | 条件分支替换 |
| Qwen3.5 MLP | `vllm/model_executor/models/qwen2.py:77-111` `Qwen2MLP` | 替换效果目标 |
| Qwen3.5 模型 | `vllm/model_executor/models/qwen3_5.py` | 架构参考 |
| OOT 注册机制 | `vllm/model_executor/custom_op.py:37-56,99-118` | PluggableLayer/CustomOp __new__ |

## 本章总结

### 我们学到了什么

1. **vLLM 的 GEMM 调用链** — 从模型层到 cuBLAS 有四层：Linear → QuantizeMethod → dispatch_gemm → F.linear

2. **三条替换路径：**
   - **路径 A（CustomOp）** — 最简单，替换激活函数等算子，一个装饰器搞定
   - **路径 B（PluggableLayer）** — 推荐，替换整个线性层，继承后覆盖 forward
   - **路径 C（GEMM 分发）** — 最底层，替换 GEMM 内核本身

3. **Qwen3.5 的特殊性** — 混合架构（full_attention + linear_attention），替换 Linear 层会同时影响两种层类型

4. **选择性替换** — 可以通过 prefix 检查只替换 MLP 或只替换 Attention 的 GEMM

5. **生产部署** — 打包为 pip 可安装的插件，通过 entry_points 自动注册

### 关键设计选择

| 选择 | 推荐 | 原因 |
|------|------|------|
| 替换粒度 | 路径 B（PluggableLayer） | 平衡了控制力和兼容性 |
| 全局 vs 选择性 | 选择性（MLP only） | Attention 的 GEMM 尺寸不同，可能需要不同优化 |
| 插件 vs 猴子补丁 | 插件 | 可维护、可版本控制、自动发现 |
| Triton vs cuBLAS | 看场景 | batch=1 cuBLAS 更快；大 batch Triton 有优化空间 |

---
← [Notebook 10: TP 通信与 GEMM 协同](10-tp-communication-gemm.ipynb)